# Collect and visualize Causal Chambers wind-tunnel measurements results

Run this notebook after any or all method notebooks in this directory. It
validates dataset and native-output hashes, converts every completed output to
the shared source-to-target orientation, and exports the inferred cross-variable
edges. Missing and failed fits remain visible in a separate status table.

This is an empirical case study and does not require graph-recovery scores.
The main output is a set of directly comparable directed
graph panels. If the dataset supplies a physical or constructed reference
network, that network appears in a separately labelled contextual panel and is
not assumed to be an exhaustive lag-resolved causal truth.

In [ ]:
from __future__ import annotations

import json
import math
import platform
import sys
from pathlib import Path

EXPECTED_PYTHON = (3, 14)
if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        "This notebook requires CPython 3.14; "
        f"found {platform.python_version()}."
    )

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
import numpy as np

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "src" / "fastCGC" / "__init__.py").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the fastC-GC project root.")
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from fastCGC.experiments.comparison_scoring import collect_dataset_graphs

In [ ]:
DATASET_KEY = "causalChamber"
METHODS = (
    "fast-cgc",
    "fast-cgc-star",
    "cgc",
    "cgc-star",
    "pcmci-plus",
    "bagged-pcmci-plus",
    "cuts-plus",
)
METHOD_LABELS = {
    "fast-cgc": "fastC-GC",
    "fast-cgc-star": "fastC-GC*",
    "cgc": "conventional c-GC",
    "cgc-star": "conventional c-GC*",
    "pcmci-plus": "PCMCI+",
    "bagged-pcmci-plus": "Bagged PCMCI+",
    "cuts-plus": "CUTS+",
}

# Collection is safe to rerun. No recovery metric is computed here.
artifacts = collect_dataset_graphs(
    PROJECT_ROOT,
    DATASET_KEY,
    methods=METHODS,
)
edge_table = artifacts.edge_table
status_table = artifacts.status_table
reference_edges = artifacts.reference_edges

print(status_table["status"].value_counts(dropna=False).to_string())
print(f"Inferred edge table: {artifacts.edge_table_path}")
print(f"Run-status table: {artifacts.status_table_path}")
print(f"Reference edge table: {artifacts.reference_edges_path}")

In [ ]:
# Choose one measured system for the figure. Set an explicit identifier to
# reproduce a different bundle; the edge tables retain every completed bundle.
PLOT_DATASET_ID: str | None = None

dataset_ids = sorted(status_table["dataset_id"].unique())
if not dataset_ids:
    raise ValueError("No empirical dataset bundles were collected.")
plot_dataset_id = PLOT_DATASET_ID or dataset_ids[0]
if plot_dataset_id not in dataset_ids:
    raise ValueError(f"Unknown PLOT_DATASET_ID: {plot_dataset_id}.")

bundle_path = (
    PROJECT_ROOT
    / "data"
    / "shared"
    / "causal-comparison"
    / DATASET_KEY
    / "datasets"
    / f"{plot_dataset_id}.npz"
)
metadata = json.loads(bundle_path.with_suffix(".json").read_text(encoding="utf-8"))
names = [str(name) for name in metadata["variable_names"]]
with np.load(bundle_path, allow_pickle=False) as stored:
    stored_positions = (
        np.asarray(stored["node_positions"], dtype=float)
        if "node_positions" in stored.files
        else None
    )


def graph_positions(node_names, positions=None):
    if positions is not None and positions.shape == (len(node_names), 2):
        if np.isfinite(positions).all() and np.ptp(positions, axis=0).max() > 0:
            centered = positions - positions.mean(axis=0, keepdims=True)
            scale = np.max(np.linalg.norm(centered, axis=1))
            if scale > 0:
                return centered / scale
    angles = np.linspace(0.0, 2.0 * np.pi, len(node_names), endpoint=False)
    return np.column_stack((np.cos(angles), np.sin(angles)))


def plot_directed_graph(ax, node_names, edges, title, positions=None):
    coordinates = graph_positions(node_names, positions)
    index = {name: position for name, position in zip(node_names, coordinates)}
    directed_pairs = set(zip(edges.get("source", []), edges.get("target", [])))
    for edge in edges.itertuples(index=False):
        start = index[str(edge.source)]
        end = index[str(edge.target)]
        reverse_exists = (str(edge.target), str(edge.source)) in directed_pairs
        score = getattr(edge, "score", np.nan)
        strength = float(score) if np.isfinite(score) else 0.65
        strength = float(np.clip(abs(strength), 0.15, 1.0))
        arrow = FancyArrowPatch(
            start,
            end,
            arrowstyle="-|>",
            mutation_scale=9,
            linewidth=0.7 + 1.3 * strength,
            color="#2f5597",
            alpha=0.35 + 0.6 * strength,
            connectionstyle=f"arc3,rad={0.13 if reverse_exists else 0.0}",
            shrinkA=14,
            shrinkB=14,
            zorder=1,
        )
        ax.add_patch(arrow)
    ax.scatter(
        coordinates[:, 0],
        coordinates[:, 1],
        s=120,
        facecolor="#f4b183",
        edgecolor="#7f6000",
        linewidth=0.8,
        zorder=2,
    )
    for name, (x_value, y_value) in zip(node_names, coordinates):
        horizontal_alignment = (
            "left" if x_value > 0.08 else "right" if x_value < -0.08 else "center"
        )
        vertical_alignment = "bottom" if y_value >= 0 else "top"
        ax.text(
            x_value * 1.08,
            y_value * 1.08,
            name,
            ha=horizontal_alignment,
            va=vertical_alignment,
            fontsize=6,
            clip_on=False,
        )
    if edges.empty:
        ax.text(
            0.5,
            0.03,
            "No selected cross-variable edges",
            transform=ax.transAxes,
            ha="center",
            va="bottom",
            fontsize=7,
            color="#666666",
        )
    ax.set_title(title, fontsize=9)
    ax.set_aspect("equal")
    ax.set_xlim(-1.25, 1.25)
    ax.set_ylim(-1.25, 1.25)
    ax.axis("off")


panels = []
reference_for_plot = reference_edges[
    reference_edges["dataset_id"] == plot_dataset_id
]
if not reference_for_plot.empty:
    panels.append(("Reference network (context only)", reference_for_plot))
for method in METHODS:
    method_edges = edge_table[
        (edge_table["dataset_id"] == plot_dataset_id)
        & (edge_table["method"] == method)
    ]
    panels.append((METHOD_LABELS[method], method_edges))

n_columns = 3
n_rows = math.ceil(len(panels) / n_columns)
figure, axes = plt.subplots(
    n_rows,
    n_columns,
    figsize=(5.4 * n_columns, 5.0 * n_rows),
    squeeze=False,
)
for axis, (title, edges) in zip(axes.flat, panels):
    plot_directed_graph(axis, names, edges, title, stored_positions)
for axis in axes.flat[len(panels) :]:
    axis.axis("off")
figure.suptitle(f"{DATASET_KEY}: {plot_dataset_id}", fontsize=12)
figure.tight_layout()

figure_dir = (
    PROJECT_ROOT
    / "data"
    / "outputs"
    / "causal-comparison"
    / DATASET_KEY
    / "graphs"
)
figure_dir.mkdir(parents=True, exist_ok=True)
pdf_path = figure_dir / f"{plot_dataset_id}-inferred-graphs.pdf"
png_path = figure_dir / f"{plot_dataset_id}-inferred-graphs.png"
figure.savefig(pdf_path, bbox_inches="tight")
figure.savefig(png_path, dpi=300, bbox_inches="tight")
plt.show()
print(pdf_path)
print(png_path)